In [22]:
import pandas as pd

# Load the CSV
df = pd.read_csv("cefr_augmented.csv")

# Check the first few rows
print(df.head())


                                                text CEFR
0  M. Meier Müllergasse 1 Stadt X Internationale ...   B2
1  Müller Julia Bahnhofsstr. 1 A Stadt X Armenien...   B2
2  Michael Meier 1 Zentralplatz 1234. Stadt X Aup...   B2
3  Eva Meier Schmidt Müllergasse 12 Stadt X Kroat...   B2
4  Abs. Frau EVA SCHMIDT BAHNHOFSTR, , 1234 STADT...   B1


## Augmentation of Dataset

In [24]:
import pandas as pd
from googletrans import Translator
import time

# Load your cleaned CSV
df = pd.read_csv("merlin_meta_german_clean.csv")

# Only minority classes
minority_df = df[df['CEFR'].isin(['A1', 'C1', 'C2'])]

translator = Translator()
augmented_texts = []

# Number of augmented versions per text
n_augment = 2  # You can change to 1,2,3

for idx, row in minority_df.iterrows():
    text = row['text']
    label = row['CEFR']
    
    for i in range(n_augment):
        try:
            # German -> English
            en_text = translator.translate(text, src='de', dest='en').text
            # English -> German
            de_text = translator.translate(en_text, src='en', dest='de').text
            augmented_texts.append({'text': de_text, 'CEFR': label})
            time.sleep(1)  # avoid hitting Google too fast
        except Exception as e:
            print(f"Translation failed for index {idx}: {e}")

# Create DataFrame of augmented texts
aug_df = pd.DataFrame(augmented_texts)

# Combine with original data
df_augmented = pd.concat([df, aug_df], ignore_index=True)

# Save to CSV
df_augmented.to_csv("cefr_augmented.csv", index=False, encoding='utf-8')
print("Augmentation done! Total rows:", len(df_augmented))


Augmentation done! Total rows: 1239


## Random Forest Classifier

In [283]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load dataset
df = pd.read_csv("cefr_augmented.csv", encoding='latin1')
df['text'] = df['text'].str.replace('Ã¼', 'ü')
df['text'] = df['text'].str.replace('Ã¤', 'ä')
df['text'] = df['text'].str.replace('Ã¶', 'ö')
df['text'] = df['text'].str.replace('ÃŸ', 'ß')

# Merge rare class (assuming C2 is sparse)
df['CEFR'] = df['CEFR'].replace({'C2': 'C1'})

# Features and labels
X = df['text']
y = df['CEFR']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF vectorizer with fixes for German
vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 3),
    sublinear_tf=True,
    lowercase=False,  # Preserve case for German nouns
    strip_accents=None  # Critical: Keep umlauts and accents intact
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Random Forest model (alternative to Logistic; tune n_estimators if needed)
model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

# Cross-validation for better accuracy estimate
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train_vec, y_train, cv=cv, scoring='accuracy')
print("Cross-Validation Accuracy Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

# Fit on full train set
model.fit(X_train_vec, y_train)

# Predictions
y_pred = model.predict(X_test_vec)

# Evaluation
print("Test Set Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred, labels=['A1', 'A2', 'B1', 'B2', 'C1']))

Cross-Validation Accuracy Scores: [0.73267327 0.74752475 0.75124378 0.74129353 0.73134328]
Mean CV Accuracy: 0.7408157233633811
Test Set Accuracy: 0.6984126984126984

Classification Report:
               precision    recall  f1-score   support

          A1       0.71      0.69      0.70        35
          A2       0.63      0.75      0.68        63
          B1       0.71      0.54      0.61        67
          B2       0.68      0.83      0.75        59
          C1       1.00      0.71      0.83        28

    accuracy                           0.70       252
   macro avg       0.74      0.70      0.71       252
weighted avg       0.71      0.70      0.70       252


Confusion Matrix:
 [[24 11  0  0  0]
 [ 9 47  6  1  0]
 [ 0 17 36 14  0]
 [ 1  0  9 49  0]
 [ 0  0  0  8 20]]


In [285]:
vectorizer = TfidfVectorizer(max_features=15000, ngram_range=(1,3))
X_train_vec = vectorizer.fit_transform(X_train)
...
model.fit(X_train_vec, y_train)


RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [287]:
def predict_cefr(text):
    # Transform the text using the previously fitted vectorizer
    text_vec = vectorizer.transform([text])
    
    # Predict CEFR level
    prediction = model.predict(text_vec)[0]
    
    return prediction


In [289]:
sample_text = """
Er hat sich stundenlang auf die Präsentation vorbereitet.
"""

print("Predicted CEFR level:", predict_cefr(sample_text))


Predicted CEFR level: A1


In [293]:
pip install xgboost textstat


   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.8/72.0 MB 2.8 MB/s eta 0:00:26
   - -------------------------------------- 1.8/72.0 MB 5.3 MB/s eta 0:00:14
   - -------------------------------------- 2.4/72.0 MB 3.4 MB/s eta 0:00:21
   - -------------------------------------- 2.4/72.0 MB 3.4 MB/s eta 0:00:21
   - -------------------------------------- 3.1/72.0 MB 2.9 MB/s eta 0:00:24
   - -------------------------------------- 3.4/72.0 MB 2.5 MB/s eta 0:00:28
   -- ------------------------------------- 3.7/72.0 MB 2.4 MB/s eta 0:00:29
   -- ------------------------------------- 4.2/72.0 MB 2.5 MB/s eta 0:00:28
   -- ------------------------------------- 4.7/72.0 MB 2.3 MB/s eta 0:00:30
   -- ------------------------------------- 4.7/72.0 MB 2.3 MB/s eta 0:00:30
   -- ------------------------------------- 5.0/72.0 MB 2.2 MB/s eta 0:00:31
   --- -----

## XGBClassifier

In [295]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import textstat  # For readability scores

# Custom transformer for linguistic features
class LinguisticFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        features = []
        for text in X:
            words = text.split()
            num_words = len(words)
            num_unique_words = len(set(words))
            avg_word_len = np.mean([len(w) for w in words]) if num_words > 0 else 0
            ttr = num_unique_words / num_words if num_words > 0 else 0
            sent_len = len(text)  # Char length as proxy for sentence complexity
            flesch = textstat.flesch_reading_ease(text)  # Adaptable to German
            features.append([num_words, num_unique_words, avg_word_len, ttr, sent_len, flesch])
        return np.array(features)

# Load dataset
df = pd.read_csv("cefr_augmented.csv", encoding='latin1')
df['text'] = df['text'].str.replace('Ã¼', 'ü')
df['text'] = df['text'].str.replace('Ã¤', 'ä')
df['text'] = df['text'].str.replace('Ã¶', 'ö')
df['text'] = df['text'].str.replace('ÃŸ', 'ß')

# Merge rare class
df['CEFR'] = df['CEFR'].replace({'C2': 'C1'})

# Features and labels
X = df[['text']]  # Use DF for column transformer
y = df['CEFR']

# Encode labels for XGBoost (needs numeric)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Pipeline: TF-IDF on text + linguistic features
preprocessor = make_column_transformer(
    (TfidfVectorizer(max_features=15000, ngram_range=(1, 3), sublinear_tf=True, lowercase=False, strip_accents=None), 'text'),
    (LinguisticFeatures(), 'text')
)

# XGBoost model
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(n_estimators=200, random_state=42, eval_metric='mlogloss', scale_pos_weight=1))  # Handles multi-class
])

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
print("Cross-Validation Accuracy Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

# Fit on full train
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_pred_labels = le.inverse_transform(y_pred)
y_test_labels = le.inverse_transform(y_test)

# Evaluation
print("Test Set Accuracy:", accuracy_score(y_test_labels, y_pred_labels))
print("\nClassification Report:\n", classification_report(y_test_labels, y_pred_labels))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_labels, y_pred_labels, labels=le.classes_))

D:\anaconda\Lib\site-packages\xgboost\training.py:199: UserWarning: [18:58:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
D:\anaconda\Lib\site-packages\xgboost\training.py:199: UserWarning: [18:59:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
D:\anaconda\Lib\site-packages\xgboost\training.py:199: UserWarning: [18:59:56] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
D:\anaconda\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:00:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
D:\anaconda\Lib\site-package

Cross-Validation Accuracy Scores: [0.7029703  0.72772277 0.68656716 0.70646766 0.70149254]
Mean CV Accuracy: 0.705044086498202


D:\anaconda\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:02:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Test Set Accuracy: 0.7341269841269841

Classification Report:
               precision    recall  f1-score   support

          A1       0.82      0.77      0.79        35
          A2       0.76      0.76      0.76        63
          B1       0.67      0.72      0.69        67
          B2       0.67      0.75      0.70        59
          C1       1.00      0.64      0.78        28

    accuracy                           0.73       252
   macro avg       0.78      0.73      0.75       252
weighted avg       0.75      0.73      0.74       252


Confusion Matrix:
 [[27  8  0  0  0]
 [ 5 48 10  0  0]
 [ 0  7 48 12  0]
 [ 1  0 14 44  0]
 [ 0  0  0 10 18]]


In [297]:
pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [303]:
!pip install datasets

   ---------------------------------------- 0.0/28.0 MB ? eta -:--:--
    --------------------------------------- 0.5/28.0 MB 2.1 MB/s eta 0:00:14
   - -------------------------------------- 1.0/28.0 MB 2.5 MB/s eta 0:00:11
   - -------------------------------------- 1.0/28.0 MB 2.5 MB/s eta 0:00:11
   -- ------------------------------------- 1.6/28.0 MB 1.8 MB/s eta 0:00:15
   -- ------------------------------------- 1.8/28.0 MB 1.8 MB/s eta 0:00:15
   -- ------------------------------------- 2.1/28.0 MB 1.7 MB/s eta 0:00:15
   --- ------------------------------------ 2.6/28.0 MB 1.6 MB/s eta 0:00:16
   --- ------------------------------------ 2.6/28.0 MB 1.6 MB/s eta 0:00:16
   ---- ----------------------------------- 2.9/28.0 MB 1.5 MB/s eta 0:00:17
   ---- ----------------------------------- 3.1/28.0 MB 1.5 MB/s eta 0:00:17
   ---- ----------------------------------- 3.1/28.0 MB 1.5 MB/s eta 0:00:17
   ---- ----------------------------------- 3.4/28.0 MB 1.3 MB/s eta 0:00:19
   ---

  You can safely remove it manually.


In [7]:
pip install --upgrade pip


   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------- ------------------

In [9]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install transformers pandas scikit-learn tqdm

In [17]:
pip install httpx==0.27.0 httpcore==1.0.5


  Attempting uninstall: h11

    Found existing installation: h11 0.9.0

    Uninstalling h11-0.9.0:

      Successfully uninstalled h11-0.9.0

   ---------------------------------------- 0/3 [h11]
   ---------------------------------------- 0/3 [h11]
   ---------------------------------------- 0/3 [h11]
  Attempting uninstall: httpcore
   ---------------------------------------- 0/3 [h11]
    Found existing installation: httpcore 0.9.1
   ---------------------------------------- 0/3 [h11]
   ------------- -------------------------- 1/3 [httpcore]
    Uninstalling httpcore-0.9.1:
   ------------- -------------------------- 1/3 [httpcore]
      Successfully uninstalled httpcore-0.9.1
   ------------- -------------------------- 1/3 [httpcore]
   ------------- -------------------------- 1/3 [httpcore]
   ------------- -------------------------- 1/3 [httpcore]
   ------------- -------------------------- 1/3 [httpcore]
  Attempting uninstall: httpx
   ------------- ------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googletrans 4.0.0rc1 requires httpx==0.13.3, but you have httpx 0.27.0 which is incompatible.
